# Notebook 5 — Class Imbalance Handling

Compares four ways of addressing the 7–10% positive rate, on one model
family, with everything inside the fold.

## What changed from R01

| Change | Reason |
|---|---|
| Resampling fitted **inside the training fold**, after in-fold preprocessing | M8 claimed this; now the code enforces it |
| No test-set scoring | GATE-1(iii) |
| Explicit verification that no synthetic row reaches a validation fold | R1 asserts "100% real data" in every evaluation partition — this makes it checkable |
| AUC-PR primary | matches M15 |
| Reweighting compared as a *fourth* option | Class weighting is the mechanism E-LightGBM's focal α implements, so it belongs in this comparison, not only in the ablation |

## Why this matters for GATE-2

M8 says the proposed model used no augmentation, and the focal objective
handles imbalance instead. Fine. But `is_unbalance=True` is honoured only by
LightGBM's built-in objectives and is **inert under a custom objective**, so
the R01 baseline received an explicit weighting correction that the proposed
arm never got. The comparison below establishes how much that correction is
worth on its own, which is what makes the ablation grid in Notebook 6b
interpretable.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      score_binary, two_level_variance)
from losses import balanced_weights

banner("NOTEBOOK 5 — CLASS IMBALANCE")
OUT = run_dir("notebook05_imbalance")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)
IMB_SEEDS = SEEDS[:3]
print(f"train_pool {train_pool.shape}  seeds {IMB_SEEDS}  |  test untouched")

try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler
    HAVE_IMB = True
except ImportError:
    HAVE_IMB = False
    print("imbalanced-learn unavailable — install it or state the omission in M8")

In [ ]:
# ---- strategies ---------------------------------------------------------
# Each takes (X_tr, y_tr) and returns (X, y, sample_weight). Only the
# TRAINING fold is ever modified; the validation fold is returned untouched.
def s_none(X, y):
    return X, y, None

def s_weight(X, y):
    return X, y, balanced_weights(y)

def s_smote(X, y):
    n_pos = int((y == 1).sum())
    k = max(1, min(5, n_pos - 1))          # k_neighbors must be < n_minority
    Xr, yr = SMOTE(random_state=SPLIT_SEED, k_neighbors=k).fit_resample(X, y)
    return pd.DataFrame(Xr, columns=X.columns), pd.Series(yr), None

def s_ros(X, y):
    Xr, yr = RandomOverSampler(random_state=SPLIT_SEED).fit_resample(X, y)
    return pd.DataFrame(Xr, columns=X.columns), pd.Series(yr), None

def s_rus(X, y):
    Xr, yr = RandomUnderSampler(random_state=SPLIT_SEED).fit_resample(X, y)
    return pd.DataFrame(Xr, columns=X.columns), pd.Series(yr), None

STRATEGIES = {"none": s_none, "class_weight": s_weight}
if HAVE_IMB:
    STRATEGIES.update({"SMOTE": s_smote, "RandomOverSample": s_ros,
                       "RandomUnderSample": s_rus})
print("strategies:", list(STRATEGIES))

In [ ]:
# ---- run ----------------------------------------------------------------
from lightgbm import LGBMClassifier
import time

rows, integrity = [], []
t0 = time.perf_counter()
for seed in IMB_SEEDS:
    for fi, (tr, vl) in enumerate(cv_splits(train_pool, seed), 1):
        X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
            train_pool.iloc[tr], train_pool.iloc[vl])
        n_val_before = len(y_vl)
        for name, fn in STRATEGIES.items():
            Xa, ya, sw = fn(X_tr, y_tr)
            model = LGBMClassifier(objective="binary", random_state=seed,
                                   **SHARED_PARAMS)
            model.fit(Xa, ya, sample_weight=sw)
            p = model.predict_proba(X_vl)[:, 1]
            rows.append({"seed": seed, "fold": fi, "arm": name,
                         "n_train_rows": len(ya),
                         "n_train_positive": int((ya == 1).sum()),
                         "train_positive_pct": round(100*float((ya==1).mean()), 1),
                         **score_binary(y_vl, p)})
            integrity.append({"seed": seed, "fold": fi, "strategy": name,
                              "val_rows_unchanged": len(y_vl) == n_val_before,
                              "val_rows": len(y_vl)})
    print(f"  seed {seed} [{time.perf_counter()-t0:.0f}s]")

fold_df = pd.DataFrame(rows)
fold_df.to_csv(OUT / "imbalance_fold_scores.csv", index=False)

# R1's claim that every evaluation partition is 100% real, made checkable
ig = pd.DataFrame(integrity)
ig.to_csv(OUT / "validation_fold_integrity.csv", index=False)
assert ig["val_rows_unchanged"].all(), "a strategy altered a validation fold"
print(f"\nINTEGRITY: all {len(ig)} (strategy, fold) pairs left the validation "
      "fold untouched. No synthetic or resampled row entered any evaluation "
      "partition. This is the check behind R1's '100% real data' sentence.")

In [ ]:
# ---- results ------------------------------------------------------------
var = two_level_variance(fold_df, PRIMARY_METRIC)
summary = (fold_df.groupby("arm")
           .agg(auc_pr=("auc_pr", "mean"), auc_roc=("auc_roc", "mean"),
                recall=("recall", "mean"), precision=("precision", "mean"),
                train_rows=("n_train_rows", "mean"),
                train_pos_pct=("train_positive_pct", "mean"))
           .reset_index()
           .merge(var[["arm", "between_seed_sd", "mean_within_seed_fold_sd"]],
                  on="arm")
           .sort_values("auc_pr", ascending=False))
summary.to_csv(OUT / "imbalance_summary.csv", index=False)
print(summary.round(4).to_string(index=False))

base = summary[summary["arm"] == "none"]["auc_pr"].iloc[0]
print(f"\nrelative to no handling ({base:.4f}):")
for _, r in summary.iterrows():
    if r["arm"] != "none":
        print(f"  {r['arm']:22s} {r['auc_pr']-base:+.4f}  "
              f"(between-seed SD {r['between_seed_sd']:.4f})")
print("\nRead the deltas against the SD, not against zero. Any delta smaller "
      "than its own between-seed SD is not a finding.")

cw = summary[summary["arm"] == "class_weight"]["auc_pr"]
if len(cw):
    print(f"\nclass weighting alone is worth {float(cw.iloc[0])-base:+.4f} AUC-PR. "
          "That is the size of the correction the R01 baseline received and the "
          "proposed arm did not — the asymmetry GATE-2 turns on.")

write_manifest(OUT, {"notebook": "05_imbalance", "test_set_scored": False,
                     "seeds": IMB_SEEDS, "strategies": list(STRATEGIES),
                     "validation_integrity_all_pass": True})